# Streaming de eventos con NASA POWER

## De una traza finita a Structured Streaming

Este notebook reúne en un solo recorrido los contenidos de I01 y E11 relacionados con streaming. Está pensado para estudiantes de distintas disciplinas y no presupone experiencia en desarrollo de software.

**Problema orientador:** una observación ocurrió el 5 de enero, pero el sistema la recibió después de avanzar hasta el final del mes. ¿Debe corregir el resultado, descartar la observación o conservarla para revisión?


## Propósito

Al finalizar podrá:

- distinguir evento, entrega, reintento, atraso y tiempo de procesamiento;
- explicar cómo ventanas, watermark y estado producen resultados incrementales;
- analizar eventos tardíos, duplicados y ventanas todavía abiertas;
- diferenciar una consulta nueva de una recuperación con el mismo checkpoint;
- construir una traza de auditoría que conecte entrada, decisión, estado y salida.


## Ruta conceptual de la sesión

Seguiremos la misma progresión de las diapositivas:

```text
1. Observar       2. Elegir       3. Agrupar       4. Retener       5. Cerrar y
   la traza          el reloj         en ventanas       el estado       recuperar
```

Primero construiremos una traza finita con pandas. Después trasladaremos las mismas preguntas a Spark Structured Streaming. El objetivo es explicar decisiones temporales, no simular una plataforma de producción.


## Glosario mínimo

- **Evento:** hecho observado, por ejemplo la precipitación del 5 de enero.
- **Entrega:** mensaje o archivo mediante el cual el evento llega al sistema.
- **Tiempo de evento:** momento en que ocurrió el hecho.
- **Tiempo de procesamiento:** momento en que el motor recibió o procesó la entrega.
- **Reintento:** nueva entrega de un evento ya enviado.
- **Ventana:** intervalo temporal usado para agrupar eventos.
- **Estado:** información que la consulta conserva entre llegadas.
- **Watermark:** referencia temporal que ayuda al motor a decidir cuánto estado debe conservar.
- **Checkpoint:** progreso y estado persistidos para continuar una consulta compatible.


# Preparación

La reproducción usa precipitación diaria de NASA POWER para enero de 2025. La fecha se utiliza como tiempo de evento a medianoche por convención didáctica. No representa la hora real de publicación del proveedor.

CHIRPS pertenece al contraste climático y espacial de E11, pero no interviene en la reproducción de eventos de este notebook.


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import subprocess
import sys

import pandas as pd
from IPython.display import display

carpeta_actual = Path.cwd().resolve()
if (carpeta_actual / '00_datos.py').exists():
    KIT = carpeta_actual
elif (carpeta_actual.parent / '00_datos.py').exists():
    KIT = carpeta_actual.parent
else:
    raise RuntimeError('Abra el notebook desde kit/ o kit/Notebooks/.')

NASA = KIT / 'data' / 'raw' / 'nasa.json'
SALIDA = KIT / 'salidas' / 'streaming'
SALIDA_I01 = SALIDA / 'I01_traza_finita'
SALIDA_I01.mkdir(parents=True, exist_ok=True)
print('Kit:', KIT)
print('Salidas:', SALIDA)


Kit: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/streaming


## Descargar y verificar NASA

El notebook comprueba primero si `nasa.json` existe. Si hace falta, descarga la copia docente; después valida su huella antes de usarla. Así se evita continuar silenciosamente con un archivo ausente o diferente.


In [6]:
def ejecutar_datos(*argumentos):
    resultado = subprocess.run(
        [sys.executable, '00_datos.py', *argumentos],
        cwd=KIT, capture_output=True, text=True
    )
    if resultado.stdout.strip():
        print(resultado.stdout.strip())
    if resultado.stderr.strip():
        print(resultado.stderr.strip())
    return resultado

if not NASA.exists():
    print('nasa.json no está disponible; se descargará la copia docente.')
    descarga = ejecutar_datos('--descargar', 'nasa')
    if descarga.returncode != 0:
        raise RuntimeError('No fue posible descargar nasa.json. Revise la conexión y vuelva a ejecutar esta celda.')

verificacion = ejecutar_datos('--verificar', 'nasa')
if verificacion.returncode != 0:
    reporte_path = KIT / 'salidas' / 'verificacion.json'
    reporte = json.loads(reporte_path.read_text(encoding='utf-8')) if reporte_path.exists() else []
    estado = reporte[0].get('estado', 'desconocido') if reporte else 'sin reporte'
    raise RuntimeError(
        f'NASA no superó la verificación (estado: {estado}). '
        f'Consulte {reporte_path} y conserve aparte cualquier versión local antes de reemplazarla.'
    )

print('Fuente NASA lista para el ejercicio.')


nasa.json no está disponible; se descargará la copia docente.
Descargando: nasa.json https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,PRECTOTCORR&community=AG&longitude=-73.36&latitude=5.54&start=20250101&end=20250131&format=JSON
Preparado: nasa.json
nasa coincide
Fuente NASA lista para el ejercicio.


# Parte I · Traza finita I01

## 1. Observar los eventos

`nasa.json` contiene una serie diaria. Cada fecha identifica una observación. Los valores `-999` representan faltantes y no se convierten en cero.


In [7]:
with NASA.open(encoding='utf-8') as archivo:
    nasa = json.load(archivo)

lluvia = pd.Series(
    nasa['properties']['parameter']['PRECTOTCORR'], name='lluvia_mm'
)
observaciones = lluvia.rename_axis('event_id').reset_index()
observaciones['fecha_evento'] = pd.to_datetime(
    observaciones['event_id'], format='%Y%m%d'
)
observaciones['lluvia_mm'] = (pd.to_numeric(observaciones['lluvia_mm'], errors='raise')
    .replace(-999, float('nan')))

assert observaciones['event_id'].is_unique
display(observaciones.head(12))


,event_id,lluvia_mm,fecha_evento
0,20250101,1.96,2025-01-01
1,20250102,13.92,2025-01-02
2,20250103,15.27,2025-01-03
3,20250104,7.43,2025-01-04
4,20250105,9.97,2025-01-05
5,20250106,2.00,2025-01-06
6,20250107,0.50,2025-01-07
7,20250108,0.28,2025-01-08
8,20250109,0.07,2025-01-09
9,20250110,0.01,2025-01-10


### Batch y streaming

El archivo NASA es finito y puede procesarse en batch. Para estudiar streaming cambiaremos el **orden de entrega**, como si pequeños archivos llegaran sucesivamente. No modificaremos los valores observados.

Un evento puede tener varias entregas. La identidad `event_id` permite reconocer que el segundo día 6 es un reintento del mismo hecho.


## 2. Elegir el reloj y construir las entregas

Primero llegan los días 1 a 4 y 6 a 12. Luego vuelve a llegar el día 6. El día 5 llega al final. `orden_entrega` representa tiempo de procesamiento de forma ordinal; `fecha_evento` conserva cuándo ocurrió la observación.


In [8]:
dias_entrega = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 6, 5]
entregas = pd.DataFrame({
    'event_id': [f'202501{dia:02d}' for dia in dias_entrega],
    'orden_entrega': range(1, len(dias_entrega) + 1),
})
entregas = entregas.merge(
    observaciones, on='event_id', how='left', validate='many_to_one', indicator=True
)
assert entregas['_merge'].eq('both').all()
assert entregas['lluvia_mm'].notna().all()
display(entregas.drop(columns='_merge'))


,event_id,orden_entrega,lluvia_mm,fecha_evento
0,20250101,1,1.96,2025-01-01
1,20250102,2,13.92,2025-01-02
2,20250103,3,15.27,2025-01-03
3,20250104,4,7.43,2025-01-04
4,20250106,5,2.00,2025-01-06
5,20250107,6,0.50,2025-01-07
6,20250108,7,0.28,2025-01-08
7,20250109,8,0.07,2025-01-09
8,20250110,9,0.01,2025-01-10
9,20250111,10,0.00,2025-01-11


### Predicción antes de programar

Complete en sus notas:

- La segunda entrega del día 6 debería clasificarse como ______ porque ______.
- El día 5 debería clasificarse como ______ si el sistema admite tres días de atraso.
- Si se aceptaran las dos entregas del día 6, la suma quedaría ______.


## 3 y 4. Clasificar y conservar estado

Usaremos una regla finita y visible:

1. si `event_id` ya fue aceptado, la nueva entrega es un duplicado;
2. si la fecha es anterior al máximo tiempo de evento observado menos la demora, se considera tardía;
3. en otro caso se acepta y su identidad entra al estado `vistos`.

Esta regla ayuda a razonar, pero no reproduce toda la semántica de Spark.


In [9]:
def clasificar_entregas(tabla, demora_dias=3):
    vistos = set()
    maximo_evento = None
    traza = []

    for entrega in tabla.itertuples(index=False):
        watermark_antes = (maximo_evento - pd.Timedelta(days=demora_dias)
                           if maximo_evento is not None else None)
        if entrega.event_id in vistos:
            estado = 'duplicado'
        elif watermark_antes is not None and entrega.fecha_evento < watermark_antes:
            estado = 'tardio'
        else:
            estado = 'aceptado'
            vistos.add(entrega.event_id)

        traza.append({
            'orden_entrega': entrega.orden_entrega,
            'event_id': entrega.event_id,
            'fecha_evento': entrega.fecha_evento,
            'lluvia_mm': entrega.lluvia_mm,
            'watermark_antes': watermark_antes,
            'estado': estado,
        })
        maximo_evento = (max(maximo_evento, entrega.fecha_evento)
                         if maximo_evento is not None else entrega.fecha_evento)

    return pd.DataFrame(traza)

traza_i01 = clasificar_entregas(entregas, demora_dias=3)
display(traza_i01)


,orden_entrega,event_id,fecha_evento,lluvia_mm,watermark_antes,estado
0,1,20250101,2025-01-01,1.96,NaT,aceptado
1,2,20250102,2025-01-02,13.92,2024-12-29,aceptado
2,3,20250103,2025-01-03,15.27,2024-12-30,aceptado
3,4,20250104,2025-01-04,7.43,2024-12-31,aceptado
4,5,20250106,2025-01-06,2.00,2025-01-01,aceptado
5,6,20250107,2025-01-07,0.50,2025-01-03,aceptado
6,7,20250108,2025-01-08,0.28,2025-01-04,aceptado
7,8,20250109,2025-01-09,0.07,2025-01-05,aceptado
8,9,20250110,2025-01-10,0.01,2025-01-06,aceptado
9,10,20250111,2025-01-11,0.00,2025-01-07,aceptado


## 5. Cerrar la traza y auditar

Una traza explica cada decisión. El control resume lo ocurrido, pero no reemplaza las filas individuales. Compararemos la suma aceptada con la suma de fechas únicas para mostrar el efecto de la política temporal.


In [10]:
aceptados = traza_i01.loc[traza_i01['estado'].eq('aceptado')]
control_i01 = {
    'entregas': len(traza_i01),
    'eventos_distintos': int(traza_i01['event_id'].nunique()),
    'aceptados': len(aceptados),
    'duplicados': int(traza_i01['estado'].eq('duplicado').sum()),
    'tardios': int(traza_i01['estado'].eq('tardio').sum()),
    'suma_aceptada_mm': float(aceptados['lluvia_mm'].sum()),
    'suma_eventos_unicos_mm': float(
        entregas.drop_duplicates('event_id')['lluvia_mm'].sum()
    ),
}
assert (control_i01['entregas'], control_i01['eventos_distintos']) == (13, 12)
assert (control_i01['aceptados'], control_i01['duplicados'], control_i01['tardios']) == (11, 1, 1)
assert abs(control_i01['suma_aceptada_mm'] - 41.44) < 1e-8

traza_i01.to_csv(SALIDA_I01 / 'traza_eventos_nasa.csv', index=False)
(SALIDA_I01 / 'control_eventos_intro.json').write_text(
    json.dumps(control_i01, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([control_i01]))


,entregas,eventos_distintos,aceptados,duplicados,tardios,suma_aceptada_mm,suma_eventos_unicos_mm
0,13,12,11,1,1,41.44,51.41


### Punto de control I01

Explique con sus palabras:

1. por qué 13 entregas representan 12 eventos;
2. por qué el día 6 no debe sumarse dos veces;
3. qué información se pierde si solo conservamos el total;
4. qué política usaría para una corrección tardía del proveedor.


# Parte II · Structured Streaming

## Del ejercicio finito al motor incremental

Spark representa la fuente como una tabla que recibe filas nuevas. `readStream` detecta archivos, el plan actualiza estado y `writeStream` publica resultados. La reproducción sigue siendo finita, pero permite observar microlotes, ventanas, watermark y checkpoint.


## Ventanas, estado y watermark

Agruparemos por ventanas fijas de siete días según `fecha`, el tiempo de evento. Una suma por ventana necesita estado porque el resultado puede cambiar con nuevas entregas.

La referencia temporal se expresa como:

```text
watermark = máximo tiempo de evento observado − demora configurada
```

La garantía funciona en una sola dirección: dentro de la demora Spark no descarta por tardanza. Más allá del umbral, un dato puede procesarse o descartarse según la operación y el estado disponible.


## Modos de salida

| Modo | Qué publica | Implicación |
|---|---|---|
| Append | filas finalizadas | una ventana puede tardar en aparecer |
| Update | filas modificadas | el destino debe admitir actualizaciones |
| Complete | tabla completa | conserva todo el estado agregado |

Usaremos `append`. Las ventanas más recientes pueden quedar pendientes si el tiempo de evento deja de avanzar.


## Preparar Spark

Antes de esta sección deben estar instalados JDK 21 y `requirements_spark.txt`. Spark se ejecuta localmente con dos hilos. Si la importación falla, revise la guía de instalación antes de cambiar el código.


In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
    .master('local[2]')
    .appName('StreamingNASAClase')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate())

print('Spark:', spark.version)
print('Python:', platform.python_version())


## Carpetas de una ejecución reproducible

Cada ejecución usa una carpeta nueva para evitar mezclar checkpoints. Dentro de la misma ejecución detendremos y reiniciaremos la consulta con el **mismo checkpoint**. Así podremos distinguir recuperación de una consulta completamente nueva.


In [ ]:
identificador = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
EJECUCION = SALIDA / 'E11_structured_streaming' / identificador
ENTRADA = EJECUCION / 'entrada'
RESULTADO = EJECUCION / 'resultado'
CHECKPOINT = EJECUCION / 'checkpoint'
ENTRADA.mkdir(parents=True, exist_ok=False)

print('Ejecución:', EJECUCION)
print('Entrada:', ENTRADA)
print('Resultado:', RESULTADO)
print('Checkpoint:', CHECKPOINT)


## Plan de la consulta

La siguiente celda define el esquema, la fuente y la suma por ventana. Es un **instrumento suministrado**. Identifique cuatro elementos: `readStream`, `withWatermark`, `window` y `writeStream`.


In [ ]:
esquema = T.StructType([
    T.StructField('event_id', T.StringType()),
    T.StructField('fecha', T.TimestampType()),
    T.StructField('lluvia_mm', T.DoubleType()),
])

flujo = spark.readStream.schema(esquema).json(str(ENTRADA))
agregado = (flujo
    .withWatermark('fecha', '3 days')
    .groupBy(F.window('fecha', '7 days'))
    .agg(F.sum('lluvia_mm').alias('lluvia_mm'),
         F.count('*').alias('observaciones')))

def iniciar_consulta():
    return (agregado.writeStream
        .format('json')
        .outputMode('append')
        .option('path', str(RESULTADO))
        .option('checkpointLocation', str(CHECKPOINT))
        .start())

agregado.explain()


## Instrumento suministrado para entregar un microlote

Cada archivo representa un grupo de observaciones que llega junto. El archivo temporal se renombra al final para que Spark no lea una escritura incompleta.


In [ ]:
lluvia_nasa = nasa['properties']['parameter']['PRECTOTCORR']

def entregar_lote(numero, dias):
    filas = []
    for dia in dias:
        event_id = f'202501{dia:02d}'
        valor = lluvia_nasa[event_id]
        if valor == -999:
            continue
        filas.append(json.dumps({
            'event_id': event_id,
            'fecha': f'2025-01-{dia:02d}T00:00:00',
            'lluvia_mm': float(valor),
        }))

    temporal = EJECUCION / f'.lote_{numero:02d}.tmp'
    destino = ENTRADA / f'lote_{numero:02d}.json'
    temporal.write_text('\n'.join(filas) + '\n', encoding='utf-8')
    temporal.replace(destino)
    print('Entregado:', destino.name, '-', len(filas), 'eventos')


## Primera ejecución

Procesaremos dos lotes y detendremos la consulta. El checkpoint debe conservar qué archivos leyó y qué ventanas permanecen abiertas.


In [ ]:
consulta_1 = iniciar_consulta()
try:
    entregar_lote(1, [1, 2, 3, 4, 6, 7, 8, 9, 10])
    consulta_1.processAllAvailable()
    entregar_lote(2, list(range(11, 21)))
    consulta_1.processAllAvailable()
    progreso_antes = list(consulta_1.recentProgress)
finally:
    consulta_1.stop()

print('Microlotes observados antes del reinicio:', len(progreso_antes))


## Recuperación con el mismo checkpoint

La consulta se inicia otra vez con el mismo plan, salida y checkpoint. Luego llegan los días 21 a 31 y, al final, el día 5. Esta secuencia constituye una prueba real de continuación dentro de la reproducción local.


In [ ]:
consulta_2 = iniciar_consulta()
try:
    entregar_lote(3, list(range(21, 32)))
    consulta_2.processAllAvailable()
    entregar_lote(4, [5])
    consulta_2.processAllAvailable()
    progreso_despues = list(consulta_2.recentProgress)
finally:
    consulta_2.stop()

print('Microlotes observados después del reinicio:', len(progreso_despues))


## Auditoría del progreso

`recentProgress` contiene evidencia operativa. Revisaremos entradas, velocidad, máximo tiempo de evento, watermark, filas en estado y registros descartados por tardanza.


In [ ]:
def resumir_progreso(etapa, progresos):
    filas = []
    for p in progresos:
        operador = (p.get('stateOperators') or [{}])[0]
        tiempo_evento = p.get('eventTime') or {}
        filas.append({
            'etapa': etapa,
            'batch_id': p.get('batchId'),
            'filas_entrada': p.get('numInputRows'),
            'filas_entrada_por_segundo': p.get('inputRowsPerSecond'),
            'filas_por_segundo': p.get('processedRowsPerSecond'),
            'duracion_trigger_ms': (p.get('durationMs') or {}).get('triggerExecution'),
            'max_event_time': tiempo_evento.get('max'),
            'watermark': tiempo_evento.get('watermark'),
            'filas_estado': operador.get('numRowsTotal'),
            'descartadas_watermark': operador.get('numRowsDroppedByWatermark', 0),
        })
    return filas

auditoria = pd.DataFrame(
    resumir_progreso('antes_reinicio', progreso_antes)
    + resumir_progreso('despues_reinicio', progreso_despues)
)
auditoria.to_csv(EJECUCION / 'auditoria_microlotes.csv', index=False)
display(auditoria)


In [ ]:
descartadas = int(auditoria['descartadas_watermark'].fillna(0).sum())
archivos_resultado = list(RESULTADO.glob('*.json'))

if archivos_resultado:
    salida_stream = spark.read.json(str(RESULTADO / '*.json'))
    salida_pandas = salida_stream.orderBy('window.start').toPandas()
else:
    salida_pandas = pd.DataFrame()

control_e11 = {
    'ejecucion': identificador,
    'checkpoint_reutilizado': True,
    'archivos_entrada': len(list(ENTRADA.glob('*.json'))),
    'filas_descartadas_por_watermark': descartadas,
    'filas_salida_publicadas': len(salida_pandas),
    'salida_append': True,
    'nota': 'Las ventanas recientes pueden permanecer pendientes.',
}
(EJECUCION / 'control_eventos_notebook.json').write_text(
    json.dumps(control_e11, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([control_e11]))
display(salida_pandas)


### Punto de control Structured Streaming

Use `auditoria_microlotes.csv`, la salida y las carpetas para responder:

1. ¿Qué máximo tiempo de evento y watermark observó antes de entregar el día 5?
2. ¿Spark registró filas descartadas por watermark?
3. ¿Qué ventanas publicó en modo append y cuáles podrían seguir pendientes?
4. ¿Qué evidencia demuestra que la segunda consulta reutilizó el checkpoint?
5. ¿Qué diferencia habría si se creara una carpeta de checkpoint nueva?


## Capacidad, confiabilidad y límites

Si la tasa de entrada supera de forma sostenida la tasa de procesamiento, la cola y el retraso crecen. Compare `filas_entrada_por_segundo`, `filas_por_segundo` y `duracion_trigger_ms`. Esta reproducción es pequeña y no constituye una prueba de backpressure, pero muestra las métricas que permitirían detectarlo.

El checkpoint del motor y la semántica del destino forman una sola historia de recuperación. El sink de archivos puede coordinarse con el checkpoint. Kafka y funciones externas pueden tener garantías diferentes; `foreachBatch` necesita una estrategia de idempotencia cuando produce efectos externos.

Esta reproducción no prueba Kafka, carga sostenida ni una operación permanente. Tampoco convierte el umbral del watermark en una regla universal de descarte. El informe debe describir lo observado en esta consulta y esta versión de Spark.


In [ ]:
spark.stop()
print('Sesión Spark cerrada.')


# Ejercicio para realizar en clase

## Cambio de política temporal

**Duración:** 35 minutos, en equipos de dos o tres personas.

La organización estudia ampliar la demora admitida de tres a siete días en la traza finita. Antes de ejecutar, cada equipo debe predecir qué ocurrirá con el día 5, el número de aceptados y la suma.

Después deben ejecutar la siguiente celda, comparar las dos políticas y explicar el costo de conservar estado durante más tiempo. Finalmente deben discutir por qué siete días todavía no bastan cuando el día 5 llega después del día 31 en la reproducción con Spark.


In [ ]:
comparaciones = []
for demora in [3, 7]:
    traza = clasificar_entregas(entregas, demora_dias=demora)
    aceptados = traza.loc[traza['estado'].eq('aceptado')]
    comparaciones.append({
        'demora_dias': demora,
        'aceptados': len(aceptados),
        'duplicados': int(traza['estado'].eq('duplicado').sum()),
        'tardios': int(traza['estado'].eq('tardio').sum()),
        'suma_aceptada_mm': float(aceptados['lluvia_mm'].sum()),
        'estado_dia_5': traza.loc[traza['event_id'].eq('20250105'), 'estado'].iloc[0],
    })

comparacion_politicas = pd.DataFrame(comparaciones)
comparacion_politicas.to_csv(SALIDA_I01 / 'comparacion_politicas.csv', index=False)
display(comparacion_politicas)


## Respuesta del equipo

Edite esta celda:

- **Predicción antes de ejecutar:** …
- **Resultado observado con tres días:** …
- **Resultado observado con siete días:** …
- **Efecto sobre estado y memoria:** …
- **Por qué la regla finita no equivale exactamente al watermark de Spark:** …
- **Política propuesta para correcciones tardías:** …


# Productos de entrega

Entregue el notebook ejecutado y las evidencias generadas:

- `I01_traza_finita/traza_eventos_nasa.csv`;
- `I01_traza_finita/control_eventos_intro.json`;
- `I01_traza_finita/comparacion_politicas.csv`;
- `E11_structured_streaming/<ejecución>/auditoria_microlotes.csv`;
- `E11_structured_streaming/<ejecución>/control_eventos_notebook.json`;
- carpetas `entrada`, `resultado` y `checkpoint`;
- respuestas de los puntos de control y del ejercicio de clase.

La conclusión debe separar lo ejecutado, lo observado y las políticas propuestas.
